# BIOT 6900 · Module 4 Demo: A Real Drug in a Predicted Pocket

**Protein:** human carbonic anhydrase II (UniProt **P00918**) · **Drug:** acetazolamide (Diamox), approved for glaucoma and altitude sickness · **Experiment:** PDB **3HS4**, 1.1 Å crystal of the enzyme bound to acetazolamide

The question: AlphaFold predicted this enzyme **without the drug and without its zinc ion**. If we place the real drug from the crystal into the predicted structure, does it fit?

Steps:
1. Download the AlphaFold model and the crystal structure (a few hundred KB).
2. Match the two sequences (the crystal uses a traditional numbering that differs from UniProt) and superimpose.
3. Find the drug in the active site and measure how it sits in the predicted pocket.
4. View it in 3D, and write PyMOL and ChimeraX scripts for a high-quality view.

Needs `numpy` and `pandas` (in `biot6900`), internet for the downloads, and `py3Dmol` for the 3D view (`pip install py3Dmol`).

In [1]:
import json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

ACC, PDB_ID, DRUG = "P00918", "3HS4", "AZM"     # AZM = acetazolamide's code in the PDB
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "structures"; OUT.mkdir(exist_ok=True)

def download(url, path):
    if path.exists() and path.stat().st_size > 0:
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "BIOT6900-course-notebook"})
    with urllib.request.urlopen(req, timeout=60) as r:
        path.write_bytes(r.read())
    return path

def fetch_alphafold_pdb(acc):
    path = OUT / f"AF-{acc}.pdb"
    if path.exists():
        return path
    urls = []
    try:
        req = urllib.request.Request(f"https://alphafold.ebi.ac.uk/api/prediction/{acc}",
                                     headers={"User-Agent": "BIOT6900-course-notebook"})
        with urllib.request.urlopen(req, timeout=60) as r:
            urls.append(json.loads(r.read())[0]["pdbUrl"])
    except Exception as e:
        print("AlphaFold API lookup failed, trying direct file names:", type(e).__name__)
    urls += [f"https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_{v}.pdb" for v in ("v6", "v4")]
    for u in urls:
        try:
            return download(u, path)
        except Exception:
            continue
    raise RuntimeError(f"Download failed. Save the PDB file from https://alphafold.ebi.ac.uk/entry/{acc} as {path}")

af_path = fetch_alphafold_pdb(ACC)
xtal_path = download(f"https://files.rcsb.org/download/{PDB_ID}.pdb", OUT / f"{PDB_ID}.pdb")
print("AlphaFold model:", af_path, "| crystal:", xtal_path)

AlphaFold model: structures/AF-P00918.pdb | crystal: structures/3HS4.pdb


In [2]:
def read_pdb(path):
    # Minimal fixed-column PDB reader (first model, first alternate location only).
    rows = []
    for line in open(path):
        if line.startswith(("ATOM", "HETATM")):
            if line[16] not in (" ", "A"):
                continue
            rows.append({"record": line[:6].strip(), "atom": line[12:16].strip(), "resname": line[17:20].strip(),
                         "chain": line[21], "resnum": int(line[22:26]), "icode": line[26].strip(),
                         "x": float(line[30:38]), "y": float(line[38:46]), "z": float(line[46:54]),
                         "b": float(line[60:66]) if line[60:66].strip() else 0.0,
                         "element": line[76:78].strip() or line[12:14].strip(), "line": line})
        elif line.startswith("ENDMDL"):
            break
    return pd.DataFrame(rows)

af = read_pdb(af_path)
xt = read_pdb(xtal_path)
xt_prot = xt[(xt.record == "ATOM") & (xt.chain == "A")]
drug_all = xt[xt.resname == DRUG]
zinc = xt[xt.resname == "ZN"]
print(f"AlphaFold model: {af.resnum.nunique()} residues, mean pLDDT {af.groupby('resnum').b.first().mean():.1f}")
print(f"Crystal chain A: {xt_prot.resnum.nunique()} residues | zinc ions: {len(zinc)} | "
      f"acetazolamide copies: {drug_all.resnum.nunique()} | waters skipped")
print("The AlphaFold model contains no zinc and no drug: it is a single protein chain.")

AlphaFold model: 260 residues, mean pLDDT 97.4
Crystal chain A: 257 residues | zinc ions: 1 | acetazolamide copies: 3 | waters skipped
The AlphaFold model contains no zinc and no drug: it is a single protein chain.


## Match the two sequences, then superimpose

The crystal uses the traditional carbonic anhydrase numbering, which doesn't match UniProt's, so we **align the sequences** to pair residues, then superimpose on confident C-alpha atoms (pLDDT ≥ 70).

In [3]:
THREE = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H",
         "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
         "TYR": "Y", "VAL": "V"}

def residues(df):
    ca = df[df.atom == "CA"].drop_duplicates(["resnum", "icode"])
    return list(zip(ca.resnum, ca.icode)), "".join(THREE.get(r, "X") for r in ca.resname), ca

def global_align(a, b, match=2, mismatch=-1, gap=-2):
    # Needleman-Wunsch; returns list of (i, j) index pairs of aligned positions
    n, m = len(a), len(b)
    S = np.zeros((n + 1, m + 1)); S[:, 0] = np.arange(n + 1) * gap; S[0, :] = np.arange(m + 1) * gap
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            S[i, j] = max(S[i-1, j-1] + (match if a[i-1] == b[j-1] else mismatch), S[i-1, j] + gap, S[i, j-1] + gap)
    i, j, pairs = n, m, []
    while i > 0 and j > 0:
        if S[i, j] == S[i-1, j-1] + (match if a[i-1] == b[j-1] else mismatch):
            pairs.append((i - 1, j - 1)); i -= 1; j -= 1
        elif S[i, j] == S[i-1, j] + gap:
            i -= 1
        else:
            j -= 1
    return pairs[::-1]

af_ids, af_seq, af_ca = residues(af)
xt_ids, xt_seq, xt_ca = residues(xt_prot)
pairs = global_align(af_seq, xt_seq)
ident = np.mean([af_seq[i] == xt_seq[j] for i, j in pairs])
print(f"Aligned {len(pairs)} residue pairs, sequence identity {ident:.0%}")

def kabsch(P, Q):
    Pc, Qc = P.mean(0), Q.mean(0)
    U, S_, Vt = np.linalg.svd((P - Pc).T @ (Q - Qc))
    d = np.sign(np.linalg.det(U @ Vt))
    R = U @ np.diag([1, 1, d]) @ Vt
    return R, Qc - Pc @ R

af_xyz = af_ca[["x", "y", "z"]].to_numpy(); xt_xyz = xt_ca[["x", "y", "z"]].to_numpy()
af_b = af_ca.b.to_numpy()
use = [(i, j) for i, j in pairs if af_seq[i] == xt_seq[j] and af_b[i] >= 70]
R, t = kabsch(af_xyz[[i for i, _ in use]], xt_xyz[[j for _, j in use]])
moved = af_xyz @ R + t
dev = np.linalg.norm(moved[[i for i, _ in use]] - xt_xyz[[j for _, j in use]], axis=1)
print(f"Superimposed on {len(use)} confident residue pairs: C-alpha RMSD {np.sqrt((dev ** 2).mean()):.2f} Å")

# Apply the same transform to every AlphaFold atom and save an aligned copy for PyMOL / ChimeraX / py3Dmol
af_all = af[["x", "y", "z"]].to_numpy() @ R + t
lines = []
for (_, row), (x, y, z) in zip(af.iterrows(), af_all):
    l = row.line
    lines.append(l[:30] + f"{x:8.3f}{y:8.3f}{z:8.3f}" + l[54:])
aligned_path = OUT / f"AF-{ACC}_aligned_to_{PDB_ID}.pdb"
aligned_path.write_text("".join(lines) + "END\n")
af[["x", "y", "z"]] = af_all
print("Saved:", aligned_path)

Aligned 257 residue pairs, sequence identity 100%
Superimposed on 257 confident residue pairs: C-alpha RMSD 0.37 Å
Saved: structures/AF-P00918_aligned_to_3HS4.pdb


## Which acetazolamide is the drug site?

The crystal contains **three** acetazolamide molecules: one in the active site, and two loosely bound on the enzyme's surface. The real drug site is the one bound to the **zinc**.

In [4]:
def min_dist(A, B):
    A = A[["x", "y", "z"]].to_numpy(); B = B[["x", "y", "z"]].to_numpy()
    return float(np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)).min())

copies = []
for rn, g in drug_all.groupby("resnum"):
    copies.append({"AZM residue": rn, "atoms": len(g), "closest distance to zinc (Å)": round(min_dist(g, zinc), 2)})
copies = pd.DataFrame(copies).sort_values("closest distance to zinc (Å)")
site_resnum = int(copies.iloc[0]["AZM residue"])
drug = drug_all[drug_all.resnum == site_resnum]
print(copies.to_string(index=False))
print(f"\nActive-site acetazolamide: residue {site_resnum} (its sulfonamide binds the zinc directly)")

 AZM residue  atoms  closest distance to zinc (Å)
         701     13                          1.94
         703     13                         16.62
         702     13                         17.31

Active-site acetazolamide: residue 701 (its sulfonamide binds the zinc directly)


## Does the real drug fit the predicted pocket?

Three checks:
1. **Clashes:** how close does the drug come to the predicted protein? Under about 2.2 Å would mean the prediction blocks the pocket.
2. **Pocket residues:** which predicted residues line the drug, and how confident is AlphaFold about them?
3. **Zinc site:** are the three zinc-binding histidines in the prediction where the crystal has them, even though the model has no zinc?

In [5]:
af_heavy = af[af.element != "H"]
clash = min_dist(drug, af_heavy)
print(f"Closest approach of the drug to any predicted protein atom: {clash:.2f} Å "
      f"({'no clash: the drug fits' if clash > 2.2 else 'clash: the predicted pocket is partly closed'})")

D = drug[["x", "y", "z"]].to_numpy()
near = []
for (rn, ic), g in af_heavy.groupby(["resnum", "icode"]):
    d = np.sqrt(((g[["x", "y", "z"]].to_numpy()[:, None, :] - D[None, :, :]) ** 2).sum(-1)).min()
    if d < 4.5:
        near.append({"AlphaFold residue": f"{g.resname.iloc[0]}{rn}", "closest to drug (Å)": round(d, 2), "pLDDT": round(g.b.iloc[0], 1)})
pocket = pd.DataFrame(near, columns=["AlphaFold residue", "closest to drug (Å)", "pLDDT"]).sort_values("closest to drug (Å)")
if len(pocket):
    print(f"\n{len(pocket)} predicted residues line the drug (within 4.5 Å); mean pLDDT {pocket.pLDDT.mean():.1f}")
else:
    print("\nNo predicted residue within 4.5 Å of the drug: check the superposition step above.")
pocket

Closest approach of the drug to any predicted protein atom: 2.58 Å (no clash: the drug fits)

13 predicted residues line the drug (within 4.5 Å); mean pLDDT 98.6


,AlphaFold residue,closest to drug (Å),pLDDT
10,THR198,2.58,98.6
0,GLN92,2.59,98.7
11,THR199,2.62,97.8
1,HIS94,2.93,98.8
9,LEU197,3.16,98.7
4,HIS119,3.19,98.9
2,HIS96,3.33,98.8
5,VAL121,3.54,98.9
6,PHE130,3.80,97.6
12,TRP208,3.93,98.9


In [6]:
# The zinc-binding histidines: in the crystal they sit about 2 Å from the zinc.
zn_lig = []
for rn, g in xt_prot[xt_prot.resname == "HIS"].groupby("resnum"):
    if min_dist(g, zinc) < 2.6:
        zn_lig.append(rn)
rows = []
for rn in zn_lig:
    j = xt_ids.index((rn, ""))
    i = next(a for a, b in pairs if b == j)
    rows.append({"crystal residue": f"HIS{rn}", "matching AlphaFold residue": f"{af_ca.resname.iloc[i]}{af_ids[i][0]}",
                 "pLDDT": round(af_b[i], 1), "C-alpha shift after superposition (Å)": round(float(np.linalg.norm(moved[i] - xt_xyz[j])), 2)})
print("Zinc ligands: crystal numbering vs UniProt numbering in the AlphaFold model")
pd.DataFrame(rows)

Zinc ligands: crystal numbering vs UniProt numbering in the AlphaFold model


,crystal residue,matching AlphaFold residue,pLDDT,C-alpha shift after superposition (Å)
0,HIS94,HIS94,98.8,0.16
1,HIS96,HIS96,98.8,0.15
2,HIS119,HIS119,98.9,0.02


**What to tell students:** if the drug shows no clash, the pocket residues have pLDDT above 90, and the zinc histidines sit within a fraction of an ångström of the crystal, then AlphaFold predicted the drug's pocket, including the arrangement of residues that hold a metal it never saw. This is the best case: a rigid, confident domain with a pre-formed pocket. The lecture's limits (one snapshot, cryptic pockets, missing metals) are about the cases where this goes wrong.

## Build the view up: prediction, crystal, then both

Three views in sequence, so students see what each structure contributes before they are combined.

These reuse `aligned_path`, `xt_prot`, `drug` and `zinc` from the cells above, so **those cells must have run in this kernel.** The check below tells you. If you restarted, use this cell's Run dropdown and pick **Run All Above**.


In [7]:
# Check the pipeline above has run in this kernel (prevents a confusing NameError)
_needed = ["aligned_path", "xt_prot", "drug", "zinc", "DRUG", "PDB_ID", "site_resnum"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    print("NOT READY - not defined yet:", ", ".join(_missing))
    print("Run the cells above first: this cell's Run dropdown -> 'Run All Above'.")
else:
    print(f"Ready: {PDB_ID} loaded, acetazolamide {site_resnum} identified, superposition applied.")
    print("Run the three views below one at a time.")

Ready: 3HS4 loaded, acetazolamide 701 identified, superposition applied.
Run the three views below one at a time.


In [8]:
# Plumbing for VSCode. py3Dmol's own .show() needs require.js, which classic Jupyter
# supplies but VSCode does not - there the viewer silently renders as nothing. These views
# are built as a self-contained iframe instead, which works in VSCode, JupyterLab and Colab.
import json as _json
from IPython.display import HTML

_CDN = "https://cdnjs.cloudflare.com/ajax/libs/3Dmol/2.5.5/3Dmol-min.js"

def show3d(models, surfaces=None, height=480, zoom_to=0, zoom=None):
    # models: [{"pdb": <text>, "styles": [(selection, style), ...]}, ...]
    js = []
    for k, m in enumerate(models):
        js.append("v.addModel(%s, 'pdb');" % _json.dumps(m["pdb"]))
        for sel, sty in m["styles"]:
            sel = dict(sel); sel["model"] = k
            js.append("v.setStyle(%s, %s);" % (_json.dumps(sel), _json.dumps(sty)))
    for opts, sel in (surfaces or []):
        js.append("v.addSurface($3Dmol.SurfaceType.VDW, %s, %s);" % (_json.dumps(opts), _json.dumps(sel)))
    js.append("v.zoomTo(%s);" % _json.dumps({"model": zoom_to}))
    if zoom:
        js.append("v.zoom(%s);" % zoom)
    js.append("v.render();")
    doc = ("<!doctype html><html><head><meta charset='utf-8'><script src='" + _CDN + "'></script></head>"
           "<body style='margin:0;background:#fff'>"
           "<div id='gl' style='width:100%;height:" + str(height) + "px;position:relative'></div>"
           "<script>var v=$3Dmol.createViewer(document.getElementById('gl'),{backgroundColor:'white'});"
           + "".join(js) + "</script></body></html>")
    doc = doc.replace("&", "&amp;").replace('"', "&quot;")
    iframe = ("<iframe srcdoc=\"" + doc + "\" style=\"width:100%;height:" + str(height)
              + "px;border:1px solid #d7dde6;border-radius:8px\"></iframe>")
    return HTML(iframe)


**1 of 3 - the AlphaFold prediction alone.** No drug, no zinc: the model is a bare protein chain. Coloured by pLDDT (blue = confident).


In [9]:
af_block = aligned_path.read_text()
print("AlphaFold model of P00918 - prediction only, nothing bound")
show3d([{"pdb": af_block,
         "styles": [({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})]}])

AlphaFold model of P00918 - prediction only, nothing bound


/Users/yajur/BIOT6900/env/lib/python3.11/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


**2 of 3 - the crystal alone.** The same enzyme solved experimentally at 1.1 A, with the real acetazolamide (magenta) and the zinc (grey sphere) the prediction does not contain.


In [10]:
xtal_block = "".join(xt_prot.line) + "".join(drug.line) + "".join(zinc.line)
print(f"Crystal {PDB_ID} chain A - with acetazolamide {site_resnum} and Zn")
show3d([{"pdb": xtal_block,
         "styles": [({}, {"cartoon": {"color": "#9aa2b1"}}),
                    ({"resn": DRUG}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.25}}),
                    ({"resn": "ZN"}, {"sphere": {"color": "grey", "radius": 0.85}})]}])

Crystal 3HS4 chain A - with acetazolamide 701 and Zn


**3 of 3 - overlaid.** Prediction (blue) on crystal (grey), using the superposition computed above. The backbones trace the same path: this is the 0.37 A RMSD as a picture.


In [11]:
print("Blue = AlphaFold prediction | grey = crystal 3HS4 | magenta = acetazolamide")
show3d([{"pdb": af_block,
         "styles": [({}, {"cartoon": {"color": "#2f7ff5"}})]},
        {"pdb": xtal_block,
         "styles": [({}, {"cartoon": {"color": "#9aa2b1"}}),
                    ({"resn": DRUG}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.25}}),
                    ({"resn": "ZN"}, {"sphere": {"color": "grey", "radius": 0.85}})]}])

Blue = AlphaFold prediction | grey = crystal 3HS4 | magenta = acetazolamide


## View it in 3D

AlphaFold model as a cartoon coloured by pLDDT (blue = confident), a see-through surface over the pocket, and the **real drug** (sticks) and **zinc** (grey sphere) from the crystal. Rotate with the mouse; scroll to zoom.

In [12]:
# The payoff view: pocket surface over the predicted residues, with the real drug and zinc
pocket_res = [int(s[3:]) for s in pocket["AlphaFold residue"]]
print(f"Pocket surface over {len(pocket_res)} predicted residues, mean pLDDT {pocket.pLDDT.mean():.1f}")
show3d([{"pdb": af_block,
         "styles": [({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})]},
        {"pdb": "".join(drug.line) + "".join(zinc.line),
         "styles": [({"resn": DRUG}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.28}}),
                    ({"resn": "ZN"}, {"sphere": {"color": "grey", "radius": 0.9}})]}],
       surfaces=[({"opacity": 0.55, "color": "white"}, {"model": 0, "resi": pocket_res})],
       zoom_to=1, zoom=0.45)

Pocket surface over 13 predicted residues, mean pLDDT 98.6


## Scripts for PyMOL and ChimeraX

These use the **already aligned** files, so they open the same scene directly. In PyMOL: `File > Run Script...` and pick the `.pml`. In ChimeraX: `open demo_chimerax.cxc` in its command line.

In [ ]:
pml = f'''# BIOT 6900 demo: acetazolamide (3HS4) in the AlphaFold model of carbonic anhydrase II
load {aligned_path.as_posix()}, af
load {xtal_path.as_posix()}, crystal
hide everything
show cartoon, af
color orange, af and b < 50
color yellow, af and b > 50 and b < 70
color lightblue, af and b > 70 and b < 90
color blue, af and b > 90
show sticks, crystal and resn {DRUG} and resi {site_resnum}
color magenta, crystal and resn {DRUG} and elem C
show spheres, crystal and resn ZN
set sphere_scale, 0.4
show surface, af within 8 of (crystal and resn {DRUG} and resi {site_resnum})
set transparency, 0.45
orient crystal and resn {DRUG} and resi {site_resnum}
zoom crystal and resn {DRUG} and resi {site_resnum}, 8
'''
cxc = f'''# BIOT 6900 demo: acetazolamide (3HS4) in the AlphaFold model of carbonic anhydrase II
open {aligned_path.resolve().as_posix()}
open {xtal_path.resolve().as_posix()}
color bfactor #1 palette alphafold
hide #2 cartoons
hide #2 atoms
show #2:{site_resnum} atoms
show #2:ZN atoms
style #2:ZN sphere
color #2:{site_resnum} magenta
surface #1
transparency #1 50
view #2:{site_resnum}
'''
Path("demo_pymol.pml").write_text(pml)
Path("demo_chimerax.cxc").write_text(cxc)
print("Wrote demo_pymol.pml and demo_chimerax.cxc")